# Plastic Reuse Engine — Model Training
Run on Google Colab with GPU runtime (Runtime → Change runtime type → T4 GPU)

In [ ]:
# ---- 1. Install dependencies ----
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
!pip install -q kaggle split-folders pillow tqdm scikit-learn

In [ ]:
# ---- 2. Download datasets ----
import os, zipfile, shutil
from pathlib import Path

BASE = Path('/content/plastic_dataset')
BASE.mkdir(exist_ok=True)

# --- TrashNet ---
!wget -q https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip -O /tmp/trashnet.zip
with zipfile.ZipFile('/tmp/trashnet.zip') as z:
    z.extractall('/tmp/trashnet')
print('TrashNet downloaded')

# --- TACO (subset via Roboflow public API) ---
!wget -q 'https://public.roboflow.com/ds/Kd9HFbKNMN?key=aJ3kqnXFQs' -O /tmp/taco.zip 2>/dev/null || echo 'TACO: use Roboflow UI'

# --- Waste Classification (Kaggle) ---
# Upload your kaggle.json first, then uncomment:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d techsash/waste-classification-data -p /tmp/
# with zipfile.ZipFile('/tmp/waste-classification-data.zip') as z:
#     z.extractall('/tmp/waste')

print('Datasets ready')

In [ ]:
# ---- 3. Build unified dataset structure ----
# Expected structure:
# plastic_dataset/
#   PET/bottle/, PET/other/
#   HDPE/bottle/, HDPE/detergent_bottle/
#   LDPE/bag/, LDPE/packaging_film/
#   PP/cap/, PP/food_container/
#   PS/tray/, PS/other/
#   PVC/other/
#   OTHER/other/
#   UNKNOWN/other/

import shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm

PLASTIC_CLASSES = ['PET','HDPE','PVC','LDPE','PP','PS','OTHER','UNKNOWN']
OBJECT_CLASSES  = ['bottle','cap','bag','tray','food_container','packaging_film','detergent_bottle','other']

# TrashNet mapping: plastic → our classes
TRASHNET_MAP = {
    'plastic': 'OTHER',   # generic plastic
    'glass':   None,
    'paper':   None,
    'metal':   None,
    'cardboard': None,
    'trash':   'UNKNOWN',
}

def copy_images(src_dir, plastic_type, object_type, dest_base):
    dest = dest_base / plastic_type / object_type
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(src_dir)
    imgs = list(src.glob('*.jpg')) + list(src.glob('*.png')) + list(src.glob('*.jpeg'))
    for img in tqdm(imgs, desc=f'{plastic_type}/{object_type}'):
        shutil.copy(img, dest / img.name)
    return len(imgs)

# Copy TrashNet plastic images
trashnet_plastic = Path('/tmp/trashnet/dataset-resized/plastic')
if trashnet_plastic.exists():
    n = copy_images(trashnet_plastic, 'OTHER', 'other', BASE)
    print(f'TrashNet plastic: {n} images')

trashnet_trash = Path('/tmp/trashnet/dataset-resized/trash')
if trashnet_trash.exists():
    n = copy_images(trashnet_trash, 'UNKNOWN', 'other', BASE)
    print(f'TrashNet trash: {n} images')

print('Dataset structure ready. Add your own photos to the appropriate folders.')
print('Then run the next cell to count images per class.')

In [ ]:
# ---- 4. Upload your own photos ----
# Use this cell to upload photos from your device
from google.colab import files
import ipywidgets as widgets

print('Instructions:')
print('1. Run this cell')
print('2. Upload your plastic photos')
print('3. Then move them to the correct class folder')
print()
print('Available classes:')
for p in PLASTIC_CLASSES:
    for o in OBJECT_CLASSES:
        d = BASE / p / o
        if d.exists():
            print(f'  {p}/{o}: {len(list(d.glob("*")))} images')

# Uncomment to upload:
# uploaded = files.upload()
# for fname, data in uploaded.items():
#     # Move to correct folder — change PET/bottle as needed
#     dest = BASE / 'PET' / 'bottle' / fname
#     dest.parent.mkdir(parents=True, exist_ok=True)
#     with open(dest, 'wb') as f: f.write(data)

In [ ]:
# ---- 5. Dataset stats ----
total = 0
for p in sorted(BASE.iterdir()):
    if p.is_dir():
        for o in sorted(p.iterdir()):
            if o.is_dir():
                n = len(list(o.glob('*.*')))
                if n > 0:
                    print(f'{p.name}/{o.name}: {n}')
                    total += n
print(f'\nTotal images: {total}')

In [ ]:
# ---- 6. Prepare flat dataset for plastic_type classification ----
# We train two separate models: one for plastic type, one for object type
import splitfolders
from pathlib import Path
import shutil

# Flat dataset: one folder per plastic type
FLAT_PLASTIC = Path('/content/flat_plastic')
FLAT_PLASTIC.mkdir(exist_ok=True)

for p in BASE.iterdir():
    if p.is_dir():
        dest = FLAT_PLASTIC / p.name
        dest.mkdir(exist_ok=True)
        for o in p.iterdir():
            if o.is_dir():
                for img in o.glob('*.*'):
                    shutil.copy(img, dest / f'{o.name}_{img.name}')

# Split train/val/test
splitfolders.ratio(str(FLAT_PLASTIC), output='/content/split_plastic',
                   seed=42, ratio=(0.7, 0.2, 0.1))
print('Split done')

In [ ]:
# ---- 7. Train MobileNetV2 for plastic type ----
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH    = 32
EPOCHS   = 30
NUM_CLASSES = len([d for d in Path('/content/split_plastic/train').iterdir() if d.is_dir()])
print(f'Classes: {NUM_CLASSES}')

# Data augmentation
train_gen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=0.2,
    height_shift_range=0.2, shear_range=0.2, zoom_range=0.2,
    horizontal_flip=True, fill_mode='nearest'
)
val_gen = ImageDataGenerator(rescale=1./255)

train_ds = train_gen.flow_from_directory('/content/split_plastic/train',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, class_mode='categorical')
val_ds = val_gen.flow_from_directory('/content/split_plastic/val',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, class_mode='categorical')

# Save class indices
import json
with open('/content/plastic_classes.json', 'w') as f:
    json.dump({v: k for k, v in train_ds.class_indices.items()}, f)
print('Class mapping:', train_ds.class_indices)

# Build model
base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False  # freeze base first

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(base.input, out)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Phase 1: train head only
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5),
]
model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

# Phase 2: fine-tune top layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

print('Training complete')

In [ ]:
# ---- 8. Evaluate ----
test_gen = ImageDataGenerator(rescale=1./255)
test_ds = test_gen.flow_from_directory('/content/split_plastic/test',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
    class_mode='categorical', shuffle=False)

loss, acc = model.evaluate(test_ds)
print(f'Test accuracy: {acc:.2%}')

# Confusion matrix
import numpy as np
from sklearn.metrics import classification_report
preds = model.predict(test_ds)
y_pred = np.argmax(preds, axis=1)
y_true = test_ds.classes
labels = list(test_ds.class_indices.keys())
print(classification_report(y_true, y_pred, target_names=labels))

In [ ]:
# ---- 9. Export to TFLite ----
# Convert with quantization for smaller size and faster inference on Android
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # int8 quantization

# Representative dataset for full int8 quantization
def representative_data_gen():
    for imgs, _ in train_ds.take(100):
        yield [imgs]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()
with open('/content/plastic_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1024 / 1024
print(f'TFLite model size: {size_mb:.2f} MB')

# Also save class labels
with open('/content/plastic_labels.txt', 'w') as f:
    for i in sorted(train_ds.class_indices, key=train_ds.class_indices.get):
        f.write(i + '\n')

print('Files ready: plastic_classifier.tflite + plastic_labels.txt')

In [ ]:
# ---- 10. Download model files ----
from google.colab import files
files.download('/content/plastic_classifier.tflite')
files.download('/content/plastic_labels.txt')
files.download('/content/plastic_classes.json')
print('Download started')